In [1]:
print(3)

3


In [2]:
import pandas as pd
import numpy as np
import json
import os

In [12]:
df = pd.read_csv("oncomarl_v2_dataset.csv")

In [13]:
df.head()
print("datset apres encoder:oncomarl_v2_dataset.csv")
df.head()

datset apres encoder:oncomarl_v2_dataset.csv


,patient_id,step,action,reward,terminated,state,next_state
0,SYN_000000,0,3,-0.067948,0,"[0.6881932616233826, 0.18603569269180298, 0.54...","[0.6940189003944397, 0.16448058187961578, 0.36..."
1,SYN_000000,1,2,-0.058004,0,"[0.6940189003944397, 0.16448058187961578, 0.36...","[0.6534637808799744, 0.22413617372512817, 0.34..."
2,SYN_000000,2,1,0.013190,0,"[0.6534637808799744, 0.22413617372512817, 0.34...","[0.6109864115715027, 0.18906739354133606, 0.23..."
3,SYN_000000,3,1,-0.023634,0,"[0.6109864115715027, 0.18906739354133606, 0.23...","[0.6158275604248047, 0.17375153303146362, 0.13..."
4,SYN_000000,4,3,0.005935,0,"[0.6158275604248047, 0.17375153303146362, 0.13...","[0.6191185116767883, 0.15529917180538177, 0.0,..."


In [14]:
df=pd.read_csv("transitions_readable.csv")
print("datset avant encoder: transitions_readable.csv" )
df.head()

datset avant encoder: transitions_readable.csv


,patient_id,step,action,reward,terminated,current_tumor_burden,current_toxicity,current_uncertainty,current_progression,current_visit_index_normalized,current_previous_action,next_tumor_burden,next_toxicity,next_uncertainty,next_progression,next_visit_index_normalized,next_previous_action,action_name,current_previous_action_name
0,SYN_000000,0,3,-0.067948,0,0.688193,0.186036,0.541963,0.0,0.000000,-1.0,0.694019,0.164481,0.364501,0.0,0.083333,3.0,MULTIDISCIPLINARY_REASSESSMENT,NONE_INITIAL_STATE
1,SYN_000000,1,2,-0.058004,0,0.694019,0.164481,0.364501,0.0,0.083333,3.0,0.653464,0.224136,0.343657,0.0,0.166667,2.0,SWITCH_PATHWAY,MULTIDISCIPLINARY_REASSESSMENT
2,SYN_000000,2,1,0.013190,0,0.653464,0.224136,0.343657,0.0,0.166667,2.0,0.610986,0.189067,0.234107,0.0,0.250000,1.0,REASSESS_CURRENT_PATHWAY,SWITCH_PATHWAY
3,SYN_000000,3,1,-0.023634,0,0.610986,0.189067,0.234107,0.0,0.250000,1.0,0.615828,0.173752,0.132253,0.0,0.333333,1.0,REASSESS_CURRENT_PATHWAY,REASSESS_CURRENT_PATHWAY
4,SYN_000000,4,3,0.005935,0,0.615828,0.173752,0.132253,0.0,0.333333,1.0,0.619119,0.155299,0.000000,0.0,0.416667,3.0,MULTIDISCIPLINARY_REASSESSMENT,REASSESS_CURRENT_PATHWAY


In [15]:
import pandas as pd
import numpy as np
import ast
import json
import argparse
import os

FEATURE_ORDER = [
    "tumor_burden",
    "toxicity",
    "uncertainty",
    "progression",
    "visit_index_normalized",
    "previous_action"
]

ACTION_MAP = {
    0: "CONTINUE_CURRENT_PATHWAY",
    1: "REASSESS_CURRENT_PATHWAY",
    2: "SWITCH_PATHWAY",
    3: "MULTIDISCIPLINARY_REASSESSMENT"
}

PREV_ACTION_MAP = {
    -1.0: "NONE_INITIAL_STATE",
    0.0: "CONTINUE_CURRENT_PATHWAY",
    1.0: "REASSESS_CURRENT_PATHWAY",
    2.0: "SWITCH_PATHWAY",
    3.0: "MULTIDISCIPLINARY_REASSESSMENT"
}

def encode_dataset(input_csv: str, output_csv: str):
    """
    تحويل أعمدة الميزات الفردية إلى مصفوفات متجهة مدمجة لـ 'state' و 'next_state'.
    """
    df = pd.read_csv(input_csv)
    
    current_cols = [f"current_{f}" for f in FEATURE_ORDER]
    next_cols = [f"next_{f}" for f in FEATURE_ORDER]
    
    # بناء مصفوفات المتجهات
    df["state"] = df[current_cols].values.tolist()
    df["next_state"] = df[next_cols].values.tolist()
    
    # تحديد الأعمدة المشفرة المطلوبة
    encoded_cols = ["patient_id", "step", "action", "reward", "terminated", "state", "next_state"]
    encoded_df = df[encoded_cols]
    
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    encoded_df.to_csv(output_csv, index=False)
    print(f"[Encoder] Converted '{input_csv}' ({len(df)} rows) -> '{output_csv}'")
    return encoded_df


def decode_dataset(input_csv: str, output_csv: str):
    """
    تفكيك متجهات 'state' و 'next_state' المدمجة وإعادتها إلى أعمدة ميزات صريحة.
    """
    df = pd.read_csv(input_csv)
    
    # تقييم السلاسل النصية للمتجهات عند تحميلها كـ text من ملف CSV
    if isinstance(df["state"].iloc[0], str):
        df["state"] = df["state"].apply(ast.literal_eval)
        df["next_state"] = df["next_state"].apply(ast.literal_eval)
        
    state_matrix = np.array(df["state"].tolist())
    next_state_matrix = np.array(df["next_state"].tolist())
    
    current_cols = [f"current_{f}" for f in FEATURE_ORDER]
    next_cols = [f"next_{f}" for f in FEATURE_ORDER]
    
    current_df = pd.DataFrame(state_matrix, columns=current_cols)
    next_df = pd.DataFrame(next_state_matrix, columns=next_cols)
    
    decoded_df = pd.concat([
        df[["patient_id", "step", "action", "reward", "terminated"]],
        current_df,
        next_df
    ], axis=1)
    
    # استعادة الأسماء الوصفية للإجراءات
    decoded_df["action_name"] = decoded_df["action"].map(ACTION_MAP)
    decoded_df["current_previous_action_name"] = decoded_df["current_previous_action"].map(PREV_ACTION_MAP)
    
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    decoded_df.to_csv(output_csv, index=False)
    print(f"[Decoder] Expanded '{input_csv}' ({len(df)} rows) -> '{output_csv}'")
    return decoded_df


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="OncoMARL v2 Dataset Encoder/Decoder Utility")
    parser.add_argument("--mode", choices=["encode", "decode"], required=True)
    parser.add_argument("--input", type=str, required=True)
    parser.add_argument("--output", type=str, required=True)
    args = parser.parse_args()
    
    if args.mode == "encode":
        encode_dataset(args.input, args.output)
    else:
        decode_dataset(args.input, args.output)

usage: ipykernel_launcher.py [-h] --mode {encode,decode} --input INPUT
                             --output OUTPUT
ipykernel_launcher.py: error: the following arguments are required: --mode, --input, --output


SystemExit: 2

/home/hicham/Desktop/project/Trip_Agent/venv_global/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:


# if __name__ == "__main__":
#     validate_and_split_dataset()

In [8]:
"""
OncoMARL v2 - Dataset Analysis
==============================

Analyses:
1. Dataset overview
2. Reward distribution
3. Reward statistics by action
4. Action distribution
5. Reward evolution by step
6. Episode length distribution
7. Terminated distribution
8. State feature distributions
9. State vs next_state comparisons
10. State transition analysis
11. Correlation between numerical variables
12. Action distribution by step
13. Cumulative reward per patient
14. Reward moving average
15. Patient trajectory examples
16. Action transition matrix

All figures are saved as PNG files.
Statistical summaries are saved as CSV files.

Expected columns:
patient_id, step, action, reward, terminated, state, next_state
"""

from pathlib import Path
import ast
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASET_PATH = Path("oncomarl_v2_dataset.csv")

OUTPUT_DIR = Path("results_dataset_analysis")
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

# Actions defined by the OncoMARL v2 environment.
ACTION_NAMES = {
    0: "CONTINUE_CURRENT_PATHWAY",
    1: "REASSESS_CURRENT_PATHWAY",
    2: "SWITCH_PATHWAY",
    3: "MULTIDISCIPLINARY_REASSESSMENT",
}

# Generic state dimensions.
# Replace these names with the actual meanings defined
# in your OncoMARL environment, if known.
STATE_NAMES = [
    "state_0",
    "state_1",
    "state_2",
    "state_3",
    "state_4",
    "state_5",
]

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def save_figure(filename):
    """Save the current figure and close it."""
    path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"[SAVED] {path}")


def parse_vector(value):
    """Convert a CSV string such as '[0.1, 0.2]' to a list."""
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float)

    if pd.isna(value):
        return np.array([], dtype=float)

    try:
        parsed = ast.literal_eval(str(value))
        return np.asarray(parsed, dtype=float).flatten()
    except (ValueError, SyntaxError, TypeError):
        return np.array([], dtype=float)


def action_name(action):
    """Return a readable action name."""
    try:
        action = int(action)
    except (ValueError, TypeError):
        return f"UNKNOWN_ACTION_{action}"

    return ACTION_NAMES.get(action, f"UNKNOWN_ACTION_{action}")


def save_table(df, filename):
    """Save a dataframe as CSV."""
    path = TABLES_DIR / filename
    df.to_csv(path, index=True)
    print(f"[SAVED] {path}")


def get_numeric_state_columns(prefix):
    """Return state feature columns for a given prefix."""
    return [
        col for col in df.columns
        if col.startswith(prefix + "_")
        and col.split("_")[-1].isdigit()
    ]


# ============================================================
# 3. LOAD DATASET
# ============================================================

print("\n" + "=" * 65)
print("ONCOMARL V2 - DATASET ANALYSIS")
print("=" * 65)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PATH.resolve()}\n"
        "Change DATASET_PATH to the location of your CSV file."
    )

df = pd.read_csv(DATASET_PATH)

required_columns = [
    "patient_id",
    "step",
    "action",
    "reward",
    "terminated",
    "state",
    "next_state",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}\n"
        f"Available columns: {df.columns.tolist()}"
    )

# Convert numerical columns.
for col in ["step", "action", "reward", "terminated"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=["patient_id", "step", "action", "reward", "terminated"]
).copy()

df["step"] = df["step"].astype(int)
df["action"] = df["action"].astype(int)
df["terminated"] = df["terminated"].astype(int)

df["action_name"] = df["action"].map(action_name)

# Parse state vectors.
df["state_vector"] = df["state"].apply(parse_vector)
df["next_state_vector"] = df["next_state"].apply(parse_vector)

valid_states = df["state_vector"].apply(len) > 0
valid_next_states = df["next_state_vector"].apply(len) > 0

print(f"\nNumber of transitions: {len(df):,}")
print(f"Number of patients: {df['patient_id'].nunique():,}")
print(f"Number of actions: {df['action'].nunique()}")
print(f"Number of valid state vectors: {valid_states.sum():,}")
print(f"Number of valid next-state vectors: {valid_next_states.sum():,}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


# ============================================================
# 4. BASIC DATASET SUMMARY
# ============================================================

summary = {
    "number_of_transitions": len(df),
    "number_of_patients": df["patient_id"].nunique(),
    "number_of_unique_actions": df["action"].nunique(),
    "number_of_terminated_transitions": int(df["terminated"].sum()),
    "mean_reward": df["reward"].mean(),
    "median_reward": df["reward"].median(),
    "std_reward": df["reward"].std(),
    "min_reward": df["reward"].min(),
    "max_reward": df["reward"].max(),
    "mean_episode_length": df.groupby("patient_id").size().mean(),
    "median_episode_length": df.groupby("patient_id").size().median(),
    "missing_values_total": int(df.isna().sum().sum()),
}

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["metric", "value"]
)

summary_df.to_csv(TABLES_DIR / "dataset_summary.csv", index=False)

print("\n--- DATASET SUMMARY ---")
for key, value in summary.items():
    print(f"{key}: {value}")


# ============================================================
# 5. ANALYSIS 1: DATASET OVERVIEW
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(
    ["Transitions", "Patients"],
    [len(df), df["patient_id"].nunique()]
)
axes[0].set_title("Dataset size")
axes[0].set_ylabel("Count")

missing = df[required_columns].isna().sum()

axes[1].bar(
    missing.index,
    missing.values
)
axes[1].set_title("Missing values by column")
axes[1].set_ylabel("Missing values")
axes[1].tick_params(axis="x", rotation=45)

save_figure("01_dataset_overview.png")


# ============================================================
# 6. ANALYSIS 2: REWARD DISTRIBUTION
# ============================================================

plt.figure()

plt.hist(
    df["reward"],
    bins=50,
    edgecolor="black",
    alpha=0.8
)

plt.axvline(
    df["reward"].mean(),
    linestyle="--",
    label=f"Mean = {df['reward'].mean():.4f}"
)

plt.axvline(
    df["reward"].median(),
    linestyle=":",
    label=f"Median = {df['reward'].median():.4f}"
)

plt.title("Distribution of rewards")
plt.xlabel("Reward")
plt.ylabel("Frequency")
plt.legend()

save_figure("02_reward_distribution.png")


# ============================================================
# 7. ANALYSIS 3: REWARD STATISTICS BY ACTION
# ============================================================

reward_by_action = df.groupby("action_name")["reward"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

save_table(reward_by_action, "reward_statistics_by_action.csv")

action_means = reward_by_action["mean"].sort_values()

plt.figure()

action_means.plot(kind="barh")

plt.axvline(0, linestyle="--")

plt.title("Mean reward by action")
plt.xlabel("Mean reward")
plt.ylabel("Action")

save_figure("03_mean_reward_by_action.png")


# Reward distributions by action.
plt.figure(figsize=(12, 6))

action_order = sorted(df["action_name"].unique())

reward_data = [
    df.loc[df["action_name"] == name, "reward"].values
    for name in action_order
]

plt.boxplot(
    reward_data,
    tick_labels=action_order,
    showfliers=False
)

plt.axhline(0, linestyle="--")
plt.title("Reward distribution by action")
plt.ylabel("Reward")
plt.xticks(rotation=30, ha="right")

save_figure("04_reward_boxplot_by_action.png")


# ============================================================
# 8. ANALYSIS 4: ACTION DISTRIBUTION
# ============================================================

action_counts = df["action_name"].value_counts()

action_percentages = (
    df["action_name"].value_counts(normalize=True) * 100
)

action_distribution = pd.DataFrame({
    "count": action_counts,
    "percentage": action_percentages,
})

save_table(action_distribution, "action_distribution.csv")

plt.figure()

action_counts.plot(kind="bar")

plt.title("Action frequency")
plt.xlabel("Action")
plt.ylabel("Number of transitions")
plt.xticks(rotation=30, ha="right")

save_figure("05_action_distribution.png")


# ============================================================
# 9. ANALYSIS 5: REWARD EVOLUTION BY STEP
# ============================================================

reward_by_step = df.groupby("step")["reward"].agg(
    ["mean", "median", "std", "count"]
)

save_table(reward_by_step, "reward_statistics_by_step.csv")

plt.figure()

plt.plot(
    reward_by_step.index,
    reward_by_step["mean"],
    label="Mean reward"
)

plt.plot(
    reward_by_step.index,
    reward_by_step["median"],
    linestyle="--",
    label="Median reward"
)

plt.axhline(0, linestyle=":")

plt.title("Reward evolution by trajectory step")
plt.xlabel("Step")
plt.ylabel("Reward")
plt.legend()

save_figure("06_reward_by_step.png")


# ============================================================
# 10. ANALYSIS 6: EPISODE LENGTH
# ============================================================

episode_lengths = df.groupby("patient_id").size()

episode_stats = episode_lengths.describe()

episode_stats.to_csv(TABLES_DIR / "episode_length_statistics.csv")

plt.figure()

plt.hist(
    episode_lengths,
    bins=40,
    edgecolor="black"
)

plt.axvline(
    episode_lengths.mean(),
    linestyle="--",
    label=f"Mean = {episode_lengths.mean():.2f}"
)

plt.title("Episode length distribution")
plt.xlabel("Number of transitions per patient")
plt.ylabel("Number of patients")
plt.legend()

save_figure("07_episode_length_distribution.png")


# ============================================================
# 11. ANALYSIS 7: TERMINATED DISTRIBUTION
# ============================================================

termination_counts = df["terminated"].value_counts().sort_index()

termination_names = {
    0: "Not terminated",
    1: "Terminated",
}

termination_counts.index = [
    termination_names.get(int(x), str(x))
    for x in termination_counts.index
]

termination_counts.to_csv(TABLES_DIR / "termination_counts.csv")

plt.figure()

termination_counts.plot(kind="bar")

plt.title("Terminated flag distribution")
plt.xlabel("Episode termination flag")
plt.ylabel("Number of transitions")
plt.xticks(rotation=0)

save_figure("08_terminated_distribution.png")


# ============================================================
# 12. ANALYSIS 8: PARSE STATE FEATURES
# ============================================================

state_df = df.loc[valid_states, "state_vector"].apply(pd.Series)
next_state_df = df.loc[valid_next_states, "next_state_vector"].apply(pd.Series)

state_dim = min(
    state_df.shape[1],
    next_state_df.shape[1]
)

if state_dim > 0:

    state_df = state_df.iloc[:, :state_dim].copy()
    next_state_df = next_state_df.iloc[:, :state_dim].copy()

    if len(STATE_NAMES) >= state_dim:
        feature_names = STATE_NAMES[:state_dim]
    else:
        feature_names = [
            f"state_{i}" for i in range(state_dim)
        ]

    state_df.columns = feature_names
    next_state_df.columns = feature_names

    # State distributions.
    state_stats = state_df.describe().T
    state_stats.to_csv(TABLES_DIR / "state_statistics.csv")

    for feature in feature_names:
        plt.figure()

        plt.hist(
            state_df[feature].dropna(),
            bins=40,
            alpha=0.75,
            edgecolor="black"
        )

        plt.title(f"Distribution of {feature}")
        plt.xlabel(feature)
        plt.ylabel("Frequency")

        save_figure(f"09_distribution_{feature}.png")

    # Compare current state and next state distributions.
    comparison = pd.DataFrame({
        "state_mean": state_df.mean(),
        "next_state_mean": next_state_df.mean(),
        "state_std": state_df.std(),
        "next_state_std": next_state_df.std(),
    })

    comparison["mean_difference"] = (
        comparison["next_state_mean"] - comparison["state_mean"]
    )

    comparison.to_csv(TABLES_DIR / "state_next_state_comparison.csv")

    comparison[["state_mean", "next_state_mean"]].plot(
        kind="bar",
        figsize=(12, 6)
    )

    plt.title("Current state vs next state")
    plt.xlabel("State feature")
    plt.ylabel("Mean value")
    plt.xticks(rotation=30, ha="right")

    save_figure("10_state_vs_next_state.png")


    # ========================================================
    # 13. ANALYSIS 9: STATE TRANSITIONS
    # ========================================================

    # Only compare pairs where both vectors are valid and dimensions match.
    valid_pairs = df[
        valid_states & valid_next_states
    ].copy()

    valid_pairs = valid_pairs[
        valid_pairs["state_vector"].apply(len) == state_dim
    ]

    valid_pairs = valid_pairs[
        valid_pairs["next_state_vector"].apply(len) == state_dim
    ]

    if not valid_pairs.empty:

        current_states = np.vstack(valid_pairs["state_vector"])
        following_states = np.vstack(valid_pairs["next_state_vector"])

        state_deltas = following_states - current_states

        delta_df = pd.DataFrame(
            state_deltas,
            columns=feature_names
        )

        delta_df.describe().T.to_csv(
            TABLES_DIR / "state_transition_statistics.csv"
        )

        # Mean state change.
        mean_deltas = delta_df.mean()

        plt.figure()

        mean_deltas.plot(kind="bar")

        plt.axhline(0, linestyle="--")

        plt.title("Mean state change: next_state - state")
        plt.xlabel("State feature")
        plt.ylabel("Mean change")
        plt.xticks(rotation=30, ha="right")

        save_figure("11_mean_state_change.png")

        # State transition distributions.
        plt.figure(figsize=(12, 6))

        plt.boxplot(
            [delta_df[col].dropna() for col in feature_names],
            tick_labels=feature_names,
            showfliers=False
        )

        plt.axhline(0, linestyle="--")

        plt.title("State transition distributions")
        plt.xlabel("State feature")
        plt.ylabel("Change")
        plt.xticks(rotation=30, ha="right")

        save_figure("12_state_transition_boxplot.png")

        # State change by action.
        valid_pairs = valid_pairs.reset_index(drop=True)

        action_delta_records = []

        for i, (_, row) in enumerate(valid_pairs.iterrows()):
            delta = (
                np.asarray(row["next_state_vector"], dtype=float)
                - np.asarray(row["state_vector"], dtype=float)
            )

            record = {
                "action_name": row["action_name"],
                "reward": row["reward"],
            }

            for j, feature in enumerate(feature_names):
                record[f"delta_{feature}"] = delta[j]

            action_delta_records.append(record)

        action_delta_df = pd.DataFrame(action_delta_records)

        delta_by_action = action_delta_df.groupby(
            "action_name"
        ).mean(numeric_only=True)

        delta_by_action.to_csv(
            TABLES_DIR / "state_change_by_action.csv"
        )

        delta_columns = [
            f"delta_{feature}" for feature in feature_names
        ]

        delta_by_action[delta_columns].plot(
            kind="bar",
            figsize=(13, 6)
        )

        plt.axhline(0, linestyle="--")

        plt.title("Mean state change by action")
        plt.xlabel("Action")
        plt.ylabel("Mean state change")
        plt.xticks(rotation=30, ha="right")

        save_figure("13_state_change_by_action.png")


# ============================================================
# 14. ANALYSIS 10: CORRELATION MATRIX
# ============================================================

numeric_cols = [
    "step",
    "action",
    "reward",
    "terminated",
]

correlation_df = df[numeric_cols].corr()

correlation_df.to_csv(TABLES_DIR / "numeric_correlations.csv")

plt.figure(figsize=(8, 6))

plt.imshow(
    correlation_df,
    aspect="auto",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="Correlation")

plt.xticks(
    range(len(correlation_df.columns)),
    correlation_df.columns,
    rotation=30,
    ha="right"
)

plt.yticks(
    range(len(correlation_df.index)),
    correlation_df.index
)

for i in range(len(correlation_df.index)):
    for j in range(len(correlation_df.columns)):
        plt.text(
            j,
            i,
            f"{correlation_df.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.title("Correlation matrix of numerical variables")

save_figure("14_correlation_matrix.png")


# ============================================================
# 15. ANALYSIS 11: ACTION DISTRIBUTION BY STEP
# ============================================================

action_step = pd.crosstab(
    df["step"],
    df["action_name"],
    normalize="index"
)

action_step.to_csv(TABLES_DIR / "action_distribution_by_step.csv")

action_step.plot(
    kind="area",
    stacked=True,
    figsize=(12, 6)
)

plt.title("Action distribution by step")
plt.xlabel("Step")
plt.ylabel("Proportion of actions")
plt.legend(title="Action", bbox_to_anchor=(1.02, 1), loc="upper left")

save_figure("15_action_distribution_by_step.png")


# ============================================================
# 16. ANALYSIS 12: CUMULATIVE REWARD PER PATIENT
# ============================================================

patient_summary = df.groupby("patient_id").agg(
    transitions=("reward", "size"),
    total_reward=("reward", "sum"),
    mean_reward=("reward", "mean"),
    terminated_transitions=("terminated", "sum"),
)

patient_summary.to_csv(TABLES_DIR / "patient_summary.csv")

plt.figure()

plt.hist(
    patient_summary["total_reward"],
    bins=40,
    edgecolor="black"
)

plt.axvline(
    patient_summary["total_reward"].mean(),
    linestyle="--",
    label=(
        f"Mean = "
        f"{patient_summary['total_reward'].mean():.3f}"
    )
)

plt.title("Cumulative reward distribution per patient")
plt.xlabel("Total reward per trajectory")
plt.ylabel("Number of patients")
plt.legend()

save_figure("16_cumulative_reward_per_patient.png")


# ============================================================
# 17. ANALYSIS 13: REWARD MOVING AVERAGE
# ============================================================

# Sort by patient and step before computing trajectory statistics.
ordered_df = df.sort_values(
    ["patient_id", "step"]
).copy()

window_size = 100

ordered_df["reward_moving_average"] = (
    ordered_df["reward"]
    .rolling(window=window_size, min_periods=1)
    .mean()
)

plt.figure(figsize=(12, 6))

plt.plot(
    np.arange(len(ordered_df)),
    ordered_df["reward_moving_average"]
)

plt.axhline(0, linestyle="--")

plt.title(
    f"Reward moving average (window = {window_size} transitions)"
)

plt.xlabel("Sorted transitions")
plt.ylabel("Moving average reward")

save_figure("17_reward_moving_average.png")


# ============================================================
# 18. ANALYSIS 14: EXAMPLE PATIENT TRAJECTORIES
# ============================================================

# Select up to five patients with the longest trajectories.
selected_patients = (
    episode_lengths
    .sort_values(ascending=False)
    .head(5)
    .index
)

trajectory_records = []

for patient_id in selected_patients:

    patient_df = ordered_df[
        ordered_df["patient_id"] == patient_id
    ].sort_values("step")

    trajectory_records.append({
        "patient_id": patient_id,
        "transitions": len(patient_df),
        "total_reward": patient_df["reward"].sum(),
        "mean_reward": patient_df["reward"].mean(),
    })

    plt.figure(figsize=(10, 5))

    plt.plot(
        patient_df["step"],
        patient_df["reward"].cumsum(),
        marker="o",
        markersize=3
    )

    plt.axhline(0, linestyle="--")

    plt.title(f"Cumulative reward trajectory: {patient_id}")
    plt.xlabel("Step")
    plt.ylabel("Cumulative reward")

    save_figure(
        f"18_trajectory_{patient_id}.png"
    )

trajectory_summary = pd.DataFrame(trajectory_records)

trajectory_summary.to_csv(
    TABLES_DIR / "example_trajectory_summary.csv",
    index=False
)


# ============================================================
# 19. ANALYSIS 15: ACTION TRANSITION MATRIX
# ============================================================

ordered_df["next_action"] = (
    ordered_df.groupby("patient_id")["action"].shift(-1)
)

valid_action_transitions = ordered_df.dropna(
    subset=["next_action"]
).copy()

if not valid_action_transitions.empty:

    valid_action_transitions["next_action"] = (
        valid_action_transitions["next_action"].astype(int)
    )

    action_transition_matrix = pd.crosstab(
        valid_action_transitions["action_name"],
        valid_action_transitions["next_action"].map(action_name),
        normalize="index"
    )

    action_transition_matrix.to_csv(
        TABLES_DIR / "action_transition_matrix.csv"
    )

    plt.figure(figsize=(9, 7))

    plt.imshow(
        action_transition_matrix.values,
        aspect="auto",
        vmin=0,
        vmax=1
    )

    plt.colorbar(label="Transition probability")

    plt.xticks(
        range(len(action_transition_matrix.columns)),
        action_transition_matrix.columns,
        rotation=30,
        ha="right"
    )

    plt.yticks(
        range(len(action_transition_matrix.index)),
        action_transition_matrix.index
    )

    for i in range(action_transition_matrix.shape[0]):
        for j in range(action_transition_matrix.shape[1]):
            plt.text(
                j,
                i,
                f"{action_transition_matrix.iloc[i, j]:.2f}",
                ha="center",
                va="center"
            )

    plt.title("Action transition matrix")

    save_figure("19_action_transition_matrix.png")


# ============================================================
# 20. EXPORT CLEANED DATASET
# ============================================================

cleaned_df = df.drop(
    columns=["state_vector", "next_state_vector"],
    errors="ignore"
)

cleaned_df.to_csv(
    OUTPUT_DIR / "dataset_cleaned.csv",
    index=False
)


# ============================================================
# 21. FINAL REPORT
# ============================================================

report = {
    "dataset_path": str(DATASET_PATH.resolve()),
    "transitions": int(len(df)),
    "patients": int(df["patient_id"].nunique()),
    "state_dimension": (
        int(state_dim) if "state_dim" in globals() else None
    ),
    "action_distribution": {
        str(k): int(v)
        for k, v in df["action_name"].value_counts().items()
    },
    "reward": {
        "mean": float(df["reward"].mean()),
        "median": float(df["reward"].median()),
        "std": float(df["reward"].std()),
        "min": float(df["reward"].min()),
        "max": float(df["reward"].max()),
    },
    "episode_length": {
        "mean": float(episode_lengths.mean()),
        "median": float(episode_lengths.median()),
        "min": int(episode_lengths.min()),
        "max": int(episode_lengths.max()),
    },
    "termination_count": int(df["terminated"].sum()),
    "figures_directory": str(FIGURES_DIR),
    "tables_directory": str(TABLES_DIR),
}

with open(
    OUTPUT_DIR / "analysis_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(report, file, indent=4, ensure_ascii=False)

print("\n" + "=" * 65)
print("ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 65)

print(f"Transitions analyzed : {len(df):,}")
print(f"Patients analyzed    : {df['patient_id'].nunique():,}")
print(f"Mean reward          : {df['reward'].mean():.6f}")
print(f"Mean episode length  : {episode_lengths.mean():.2f}")
print(f"Figures saved in     : {FIGURES_DIR.resolve()}")
print(f"Tables saved in      : {TABLES_DIR.resolve()}")
print(f"JSON report          : {OUTPUT_DIR / 'analysis_report.json'}")

print("\nGenerated figures:")
for figure_path in sorted(FIGURES_DIR.glob("*.png")):
    print(f" - {figure_path.name}")

print("\nAll analyses finished.")


ONCOMARL V2 - DATASET ANALYSIS

Number of transitions: 120,000
Number of patients: 10,000
Number of actions: 4
Number of valid state vectors: 120,000
Number of valid next-state vectors: 120,000
Output directory: /home/hicham/Desktop/project/Trip_Agent/OncoMARL_v2/data/raw/results_dataset_analysis

--- DATASET SUMMARY ---
number_of_transitions: 120000
number_of_patients: 10000
number_of_unique_actions: 4
number_of_terminated_transitions: 10000
mean_reward: -0.014180402582322109
median_reward: -0.0084016902976619
std_reward: 0.037418045379477946
min_reward: -0.1882890143541561
max_reward: 0.1013960928548481
mean_episode_length: 12.0
median_episode_length: 12.0
missing_values_total: 0
[SAVED] results_dataset_analysis/figures/01_dataset_overview.png
[SAVED] results_dataset_analysis/figures/02_reward_distribution.png
[SAVED] results_dataset_analysis/tables/reward_statistics_by_action.csv
[SAVED] results_dataset_analysis/figures/03_mean_reward_by_action.png
[SAVED] results_dataset_analysis/

In [7]:
%pip install matplotlib

  Using cached matplotlib-3.11.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached contourpy-1.4.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.67.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (137 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 kB)
Using cached matplotlib-3.11.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.9 MB)
Using cached contourpy-1.4.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (383 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 1.1 MB/s eta 0:00:0000:0100:010m
Using cached kiwisolver-1.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (1.5 MB)
Note: you may need to restart the kernel to use updated packages

In [9]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


# =====================================================
# CONFIGURATION
# =====================================================

JSON_PATH = Path("train_summary_full.json")
OUTPUT_DIR = Path("training_results")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# LOAD RESULTS
# =====================================================

if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {JSON_PATH.resolve()}"
    )

with open(JSON_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

if isinstance(results, dict):
    # Supporte aussi un JSON contenant une clé de résultats.
    if "results" in results:
        results = results["results"]
    else:
        results = [results]

if not results:
    raise ValueError("Aucun résultat d'entraînement trouvé.")

df = pd.DataFrame([
    {
        "seed": item.get("seed"),
        "mean_final_reward": item.get("mean_final_reward"),
        "mean_agreement": item.get("mean_agreement"),
    }
    for item in results
])

print("\n===== TRAINING SUMMARY =====")
print(df.to_string(index=False))

df.to_csv(OUTPUT_DIR / "training_summary.csv", index=False)


# =====================================================
# 1. COMPARISON OF FINAL REWARDS
# =====================================================

plt.figure(figsize=(10, 6))

plt.bar(
    df["seed"].astype(str),
    df["mean_final_reward"]
)

plt.axhline(0, linestyle="--")

plt.title("Mean Final Reward by Seed")
plt.xlabel("Seed")
plt.ylabel("Mean Final Reward")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "01_final_reward_by_seed.png",
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# =====================================================
# 2. COMPARISON OF AGREEMENT
# =====================================================

plt.figure(figsize=(10, 6))

plt.bar(
    df["seed"].astype(str),
    df["mean_agreement"]
)

plt.title("Mean Agreement by Seed")
plt.xlabel("Seed")
plt.ylabel("Mean Agreement")
plt.ylim(0, 1)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "02_agreement_by_seed.png",
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# =====================================================
# 3. REWARD HISTORY OF EACH SEED
# =====================================================

for item in results:

    seed = item.get("seed", "unknown")
    reward_history = item.get("reward_history", [])

    if not reward_history:
        print(f"[SKIP] Seed {seed}: no reward_history")
        continue

    reward_history = np.asarray(
        reward_history,
        dtype=float
    )

    plt.figure(figsize=(12, 6))

    plt.plot(
        np.arange(1, len(reward_history) + 1),
        reward_history,
        label=f"Seed {seed}"
    )

    plt.axhline(0, linestyle="--")

    plt.title(f"Training Reward History - Seed {seed}")
    plt.xlabel("Training Episode / Recorded Iteration")
    plt.ylabel("Reward")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"03_reward_history_seed_{seed}.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()


# =====================================================
# 4. REWARD HISTORIES COMPARISON
# =====================================================

plt.figure(figsize=(12, 6))

for item in results:

    seed = item.get("seed", "unknown")
    reward_history = item.get("reward_history", [])

    if len(reward_history) == 0:
        continue

    plt.plot(
        np.arange(1, len(reward_history) + 1),
        reward_history,
        label=f"Seed {seed}",
        alpha=0.8
    )

plt.axhline(0, linestyle="--")

plt.title("Reward History Comparison Across Seeds")
plt.xlabel("Training Episode / Recorded Iteration")
plt.ylabel("Reward")
plt.legend()
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "04_all_reward_histories.png",
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# =====================================================
# 5. SMOOTHED REWARD HISTORY
# =====================================================

for item in results:

    seed = item.get("seed", "unknown")
    rewards = np.asarray(
        item.get("reward_history", []),
        dtype=float
    )

    if len(rewards) == 0:
        continue

    window = min(10, len(rewards))

    smoothed = pd.Series(rewards).rolling(
        window=window,
        min_periods=1
    ).mean()

    plt.figure(figsize=(12, 6))

    plt.plot(
        np.arange(1, len(rewards) + 1),
        rewards,
        alpha=0.35,
        label="Original reward"
    )

    plt.plot(
        np.arange(1, len(rewards) + 1),
        smoothed,
        label=f"Moving average ({window})"
    )

    plt.axhline(0, linestyle="--")

    plt.title(f"Smoothed Training Rewards - Seed {seed}")
    plt.xlabel("Training Episode / Recorded Iteration")
    plt.ylabel("Reward")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"05_smoothed_reward_seed_{seed}.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()


# =====================================================
# 6. FINAL STATISTICS
# =====================================================

statistics = df[
    ["mean_final_reward", "mean_agreement"]
].describe()

statistics.to_csv(
    OUTPUT_DIR / "training_statistics.csv"
)

print("\n===== GENERATED FILES =====")

for path in sorted(OUTPUT_DIR.iterdir()):
    print(path)

print("\nAnalysis completed successfully.")
print(f"Results saved in: {OUTPUT_DIR.resolve()}")


===== TRAINING SUMMARY =====
 seed  mean_final_reward  mean_agreement
   10           0.567714        0.885000
   20           0.599225        0.888333
   30           0.594234        0.865000
   40           0.606581        0.876667
   50           0.598140        0.873333

===== GENERATED FILES =====
training_results/01_final_reward_by_seed.png
training_results/02_agreement_by_seed.png
training_results/03_reward_history_seed_10.png
training_results/03_reward_history_seed_20.png
training_results/03_reward_history_seed_30.png
training_results/03_reward_history_seed_40.png
training_results/03_reward_history_seed_50.png
training_results/04_all_reward_histories.png
training_results/05_smoothed_reward_seed_10.png
training_results/05_smoothed_reward_seed_20.png
training_results/05_smoothed_reward_seed_30.png
training_results/05_smoothed_reward_seed_40.png
training_results/05_smoothed_reward_seed_50.png
training_results/training_statistics.csv
training_results/training_summary.csv

Analysis

In [10]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


# =====================================================
# CONFIGURATION
# =====================================================

JSON_PATH = Path("test_evaluation_summary.json")
OUTPUT_DIR = Path("test_evaluation_results")
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


# =====================================================
# LOAD JSON
# =====================================================

if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {JSON_PATH.resolve()}"
    )

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Type de données :", type(data).__name__)

if isinstance(data, list):
    records = data

elif isinstance(data, dict):
    # Si le JSON contient une liste sous une clé spécifique.
    records = None

    for key, value in data.items():
        if isinstance(value, list) and value:
            if all(isinstance(item, dict) for item in value):
                records = value
                print(f"Liste de résultats détectée : {key}")
                break

    if records is None:
        # Un objet JSON unique contenant les métriques.
        records = [data]

else:
    raise ValueError("Structure JSON non prise en charge.")

if not records:
    raise ValueError("Aucun résultat trouvé dans le fichier JSON.")

print("\nNombre d'enregistrements :", len(records))
print("\nClés disponibles :")

all_keys = sorted({
    key
    for record in records
    for key in record.keys()
})

for key in all_keys:
    print(" -", key)


# =====================================================
# CONVERT JSON RECORDS TO DATAFRAME
# =====================================================

df = pd.json_normalize(records, sep="_")

# Ajouter une colonne seed si elle existe.
if "seed" not in df.columns:
    df["seed"] = np.arange(1, len(df) + 1)

# Exporter les données tabulaires.
df.to_csv(
    TABLES_DIR / "evaluation_summary.csv",
    index=False
)

# Détecter les colonnes numériques.
numeric_columns = [
    col
    for col in df.select_dtypes(include=np.number).columns
    if col != "seed"
]

print("\nMétriques numériques détectées :")
for column in numeric_columns:
    print(" -", column)


# =====================================================
# 1. COMPARISON OF METRICS ACROSS SEEDS
# =====================================================

for metric in numeric_columns:

    values = pd.to_numeric(df[metric], errors="coerce")
    valid = values.notna()

    if not valid.any():
        continue

    plot_df = df.loc[valid].copy()
    plot_df[metric] = values.loc[valid]

    plt.figure(figsize=(10, 6))

    plt.bar(
        plot_df["seed"].astype(str),
        plot_df[metric]
    )

    plt.title(f"{metric} by Seed")
    plt.xlabel("Seed")
    plt.ylabel(metric)
    plt.axhline(0, linestyle="--", linewidth=0.8)

    plt.tight_layout()

    safe_name = "".join(
        c if c.isalnum() or c in "-_" else "_"
        for c in metric
    )

    plt.savefig(
        FIGURES_DIR / f"metric_{safe_name}_by_seed.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()


# =====================================================
# 2. DISTRIBUTION OF EACH NUMERICAL METRIC
# =====================================================

for metric in numeric_columns:

    values = pd.to_numeric(
        df[metric],
        errors="coerce"
    ).dropna()

    if values.empty:
        continue

    plt.figure(figsize=(9, 5))

    plt.hist(
        values,
        bins=min(20, max(5, len(values))),
        edgecolor="black"
    )

    plt.title(f"Distribution of {metric}")
    plt.xlabel(metric)
    plt.ylabel("Frequency")

    plt.tight_layout()

    safe_name = "".join(
        c if c.isalnum() or c in "-_" else "_"
        for c in metric
    )

    plt.savefig(
        FIGURES_DIR / f"distribution_{safe_name}.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()


# =====================================================
# 3. REWARD HISTORY, IF AVAILABLE
# =====================================================

history_columns = [
    col for col in df.columns
    if "reward_history" in col.lower()
]

for history_col in history_columns:

    plt.figure(figsize=(12, 6))
    plotted = False

    for _, row in df.iterrows():

        history = row[history_col]

        if not isinstance(history, (list, np.ndarray)):
            continue

        if len(history) == 0:
            continue

        history = np.asarray(history, dtype=float)

        seed = row["seed"]

        plt.plot(
            np.arange(1, len(history) + 1),
            history,
            label=f"Seed {seed}",
            alpha=0.8
        )

        plotted = True

    if plotted:

        plt.title("Evaluation Reward History")
        plt.xlabel("Evaluation Step")
        plt.ylabel("Reward")
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            FIGURES_DIR / "reward_history_comparison.png",
            dpi=200,
            bbox_inches="tight"
        )

    plt.close()


# =====================================================
# 4. SAVE DESCRIPTIVE STATISTICS
# =====================================================

if numeric_columns:

    stats = df[numeric_columns].describe().T

    stats.to_csv(
        TABLES_DIR / "evaluation_statistics.csv"
    )

    print("\n===== EVALUATION STATISTICS =====")
    print(stats.to_string())


# =====================================================
# 5. SAVE A SIMPLE TEXT REPORT
# =====================================================

report_path = OUTPUT_DIR / "evaluation_report.txt"

with open(report_path, "w", encoding="utf-8") as f:

    f.write("TEST EVALUATION REPORT\n")
    f.write("=" * 50 + "\n\n")

    f.write(f"Number of records: {len(df)}\n")
    f.write(f"Number of numerical metrics: {len(numeric_columns)}\n\n")

    f.write("Numerical metrics:\n")

    for metric in numeric_columns:

        values = pd.to_numeric(
            df[metric],
            errors="coerce"
        ).dropna()

        if values.empty:
            continue

        f.write(f"\n{metric}\n")
        f.write(f"  Mean:   {values.mean():.6f}\n")
        f.write(f"  Median: {values.median():.6f}\n")
        f.write(f"  Std:    {values.std():.6f}\n")
        f.write(f"  Min:    {values.min():.6f}\n")
        f.write(f"  Max:    {values.max():.6f}\n")


# =====================================================
# 6. FINISH
# =====================================================

print("\n" + "=" * 60)
print("TEST EVALUATION ANALYSIS COMPLETED")
print("=" * 60)

print(f"JSON source: {JSON_PATH.resolve()}")
print(f"Figures:     {FIGURES_DIR.resolve()}")
print(f"Tables:      {TABLES_DIR.resolve()}")
print(f"Report:      {report_path.resolve()}")

print("\nGenerated files:")

for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(" -", path)

Type de données : dict
Liste de résultats détectée : per_example_results

Nombre d'enregistrements : 18000

Clés disponibles :
 - example_index
 - is_unanimous
 - patient_id
 - prediction_details
 - voted_action_id
 - voted_action_name

Métriques numériques détectées :
 - example_index
 - voted_action_id
 - prediction_details_voted_action_id

===== EVALUATION STATISTICS =====
                                      count         mean          std  min      25%     50%       75%      max
example_index                       18000.0  8999.500000  5196.296758  0.0  4499.75  8999.5  13499.25  17999.0
voted_action_id                     18000.0     0.440389     0.496448  0.0     0.00     0.0      1.00      1.0
prediction_details_voted_action_id  18000.0     0.440389     0.496448  0.0     0.00     0.0      1.00      1.0

TEST EVALUATION ANALYSIS COMPLETED
JSON source: /home/hicham/Desktop/project/Trip_Agent/OncoMARL_v2/data/raw/test_evaluation_summary.json
Figures:     /home/hicham/Desktop/proj